In [ ]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.messages import RemoveMessage # for permanent deletion of messages from state

In [ ]:
load_dotenv()

In [ ]:
model = ChatGroq(model="moonshotai/kimi-k2-instruct-0905")

In [ ]:
def chat(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def delete_old_messages(state: MessagesState):
    msgs = state["messages"]

    # if more than 10 messages, delete the earliest 6
    if len(msgs) > 10:
        to_remove = msgs[:6]
        return {"messages": [RemoveMessage(id=m.id) for m in to_remove]}

    return {}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("chat", chat)
builder.add_node("cleanup", delete_old_messages)

In [ ]:
builder.add_edge(START, "chat")
builder.add_edge("chat", "cleanup")   # run deletion after each response
builder.add_edge("cleanup", "__end__")

In [ ]:
graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
graph

In [ ]:
config = {"configurable": {"thread_id": "t1"}}

In [ ]:
# Run multiple turns
# After all the execution we will be having 14 messages in state (7 for AI and & for Human)
graph.invoke({"messages": [{"role": "user", "content": "Hi, I'm Irfan"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Tell me about LangGraph"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "Now explain checkpointers"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Langchain"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Quantum Mechanics"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is Gen AI"}]}, config)
graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config)

In [ ]:
# Since total messages = 14 which is > 10 so the oldest (starting) 6 messages will be deleted
snap = graph.get_state(config)
print("Stored messages after cleanup:", len(snap.values["messages"]))